# AML Data Preprocessing

What this notebook does:
1. Loads the raw AML transaction CSV
2. Parses Timestamp from date-hour-minute to relative seconds
3. Label-encodes Payment Format, Receiving Currency, and Payment Currency as integers (shared dict for currencies, same as original)
4. Merges From Bank + Account / To Bank + Account to unique integer node IDs (from_id, to_id) via a shared account dict
5. Adds EdgeID (sequential row index)
6. Renames / drops raw columns to match the formatted output schema
7. Sorts by Timestamp (matching the original sort(3))
8. Saves the result as formatted_transactions.csv

1. Load the raw dataset

In [1]:
import os
import sys
import pandas as pd
from datetime import datetime

sys.path.insert(0, os.path.dirname(os.path.abspath('get_dataset.py')))
from get_dataset import load_aml_dataset

df = load_aml_dataset()
print(f"Loaded {len(df)} rows")
print(f"Columns : {df.columns.tolist()}")

Loaded 15000000 rows
Columns : ['Timestamp', 'From Bank', 'Account', 'To Bank', 'Account.1', 'Amount Received', 'Receiving Currency', 'Amount Paid', 'Payment Currency', 'Payment Format', 'Is Laundering']


2. Parse Timestamp from date-hour-minute to relative seconds

In [2]:
dt_series = pd.to_datetime(df["Timestamp"], format="%Y/%m/%d %H:%M")

first_dt   = dt_series.min()
start_time = datetime(first_dt.year, first_dt.month, first_dt.day)
firstTs    = start_time.timestamp() - 10

df["Timestamp"] = (dt_series.astype("int64") // 10**9 - firstTs).astype("int64")

print(f"Reference epoch (firstTs): {firstTs}")
range_buffer = df["Timestamp"].max() - df["Timestamp"].min()
print(f"Timestamp range: {range_buffer} seconds")
df[["Timestamp"]].head(3)

Reference epoch (firstTs): 1659292190.0
Timestamp range: 6430860 seconds


,Timestamp
0,20830
1,19930
2,20830


3. Label-Encode Categorical Columns

Label encoding done for the payment type and payment currency

In [3]:
def encode_columns_shared(df, cols) :
    shared = {}
    result = {}
    for col in cols :
        encoded = []
        for val in df[col] :
            if val not in shared :
                shared[val] = len(shared)
            encoded.append(shared[val])
        result[col] = encoded
    return result, shared

def encode_column(series):
    label_dict = {}
    encoded = []
    for val in series:
        if val not in label_dict:
            label_dict[val] = len(label_dict)
        encoded.append(label_dict[val])
    return encoded, label_dict

currency_encoded, currency_dict = encode_columns_shared(df, ['Receiving Currency', 'Payment Currency'])
df['Receiving Currency'] = currency_encoded['Receiving Currency']
df['Payment Currency']   = currency_encoded['Payment Currency']

fmt_encoded, fmt_dict = encode_column(df['Payment Format'])
df['Payment Format'] = fmt_encoded

print("Currency encoding : ", currency_dict)
print("Payment Format enc : ", fmt_dict)
df[['Receiving Currency', 'Payment Currency', 'Payment Format']].head(3)

Currency encoding :  {'US Dollar': 0, 'Euro': 1, 'UK Pound': 2, 'Bitcoin': 3, 'Yen': 4, 'Yuan': 5, 'Canadian Dollar': 6, 'Rupee': 7, 'Australian Dollar': 8, 'Ruble': 9, 'Shekel': 10, 'Brazil Real': 11, 'Mexican Peso': 12, 'Swiss Franc': 13, 'Saudi Riyal': 14}
Payment Format enc :  {'Reinvestment': 0, 'Cheque': 1, 'Credit Card': 2, 'ACH': 3, 'Wire': 4, 'Cash': 5, 'Bitcoin': 6}


,Receiving Currency,Payment Currency,Payment Format
0,0,0,0
1,0,0,0
2,0,0,0


4. Build from_id and to_id Node IDs

Concatenating the From Bank + Account and To Bank + Account as strings and then assigning IDs from a single shared account dict (so the same account always gets the same integer regardless of whether it is sender or receiver)

In [4]:
account_dict = {}

def get_account_id(bank, acc) :
    key = str(bank) + str(acc)
    if key not in account_dict:
        account_dict[key] = len(account_dict)
    return account_dict[key]

to_acc_col = 'Account.1' if 'Account.1' in df.columns else df.columns[4]

df['from_id'] = [get_account_id(b, a) for b, a in zip(df['From Bank'], df['Account'])]
df['to_id'] = [get_account_id(b, a) for b, a in zip(df['To Bank'], df[to_acc_col])]

print(f"Total unique accounts (nodes) : {len(account_dict)}")
df[['From Bank', 'Account', 'from_id', 'To Bank', to_acc_col, 'to_id']].head(3)

Total unique accounts (nodes) : 2061626


,From Bank,Account,from_id,To Bank,Account.1,to_id
0,20,800104D70,0,20,800104D70,0
1,3196,800107150,1,3196,800107150,1
2,1208,80010E430,2,1208,80010E430,2


5. Add EdgeID, Rename and Drop Columns

Final schema:
EdgeID, from_id, to_id, Timestamp, Amount Sent, Sent Currency, Amount Received, Received Currency, Payment Format, Is Laundering

In [5]:
df.insert(0, 'EdgeID', range(len(df)))

df.rename(columns={'Amount Paid': 'Amount Sent', 'Payment Currency': 'Sent Currency', 'Receiving Currency': 'Received Currency'}, inplace=True)

cols_to_drop = ['From Bank', 'Account', 'To Bank', to_acc_col]
df.drop(columns=cols_to_drop, inplace=True)

df = df[['EdgeID', 'from_id', 'to_id', 'Timestamp', 'Amount Sent', 'Sent Currency', 'Amount Received', 'Received Currency', 'Payment Format', 'Is Laundering']]

df['EdgeID'] = df['EdgeID'].astype('int32')
df['from_id'] = df['from_id'].astype('int32')
df['to_id'] = df['to_id'].astype('int32')
df['Timestamp'] = df['Timestamp'].astype('int32')
df['Amount Sent'] = df['Amount Sent'].astype('float32')
df['Sent Currency'] = df['Sent Currency'].astype('int16')
df['Amount Received'] = df['Amount Received'].astype('float32')
df['Received Currency'] = df['Received Currency'].astype('int16')
df['Payment Format'] = df['Payment Format'].astype('int8')
df['Is Laundering'] = df['Is Laundering'].astype('int8')

print(f"Memory usage optimized: {df.memory_usage().sum() / 1e6:.1f} MB")
print("Columns:", df.columns.tolist())
df.head(3)

Memory usage optimized: 450.0 MB
Columns: ['EdgeID', 'from_id', 'to_id', 'Timestamp', 'Amount Sent', 'Sent Currency', 'Amount Received', 'Received Currency', 'Payment Format', 'Is Laundering']


,EdgeID,from_id,to_id,Timestamp,Amount Sent,Sent Currency,Amount Received,Received Currency,Payment Format,Is Laundering
0,0,0,0,20830,6794.629883,0,6794.629883,0,0,0
1,1,1,1,19930,7739.290039,0,7739.290039,0,0,0
2,2,2,2,20830,1880.229980,0,1880.229980,0,0,0


6. Sort by Timestamp



In [7]:
df.sort_values('Timestamp', inplace = True)
df.reset_index(drop = True, inplace = True)

print(f"Shape : {df.shape}")
print(f"Timestamp range : {range_buffer} seconds")
print(f"Laundering done : {(df['Is Laundering'] == 1).sum()}")
print(f"Laundering not done : {(df['Is Laundering'] == 0).sum()}")
print(f"Illicit ratio : {df['Is Laundering'].sum():,} / {len(df):,} = {df['Is Laundering'].mean()*100:.3f}%")
df.head(5)

Shape : (15000000, 10)
Timestamp range : 6430860 seconds
Laundering done : 15342
Laundering not done : 14984658
Illicit ratio : 15,342 / 15,000,000 = 0.102%


,EdgeID,from_id,to_id,Timestamp,Amount Sent,Sent Currency,Amount Received,Received Currency,Payment Format,Is Laundering
0,512274,382542,382542,19810,21.330000,1,21.330000,1,0,0
1,899651,672087,672087,19810,285062.937500,4,285062.937500,4,0,0
2,1108640,813025,827655,19810,5933.709961,8,5933.709961,8,2,0
3,536358,400304,400304,19810,17.290001,1,17.290001,1,0,0
4,700502,523060,488519,19810,11.820000,1,11.820000,1,2,0


7. Save Formatted Transactions

In [8]:
out_path = "formatted_transactions.csv"

df.to_csv(out_path, index=False)
print(f"Saved  → {os.path.abspath(out_path)}")
print(f"Size   : {os.path.getsize(out_path) / 1e9:.2f} GB")

Saved  → /home/shreyas-nalle/Desktop/Delusional/model/formatted_transactions.csv
Size   : 0.80 GB
